[Documentation](https://developer.nvidia.com/blog/enhancing-rag-pipelines-with-re-ranking/)

In [1]:
from langchain_community.document_loaders import PyPDFLoader
document = PyPDFLoader("Data/VILA.pdf").load()

In [2]:
len(document)

16

In [3]:
# split the documents into chunks
from langchain_text_splitters  import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)
chunks = splitter.split_documents(documents = document)

In [4]:
len(chunks)

175

In [5]:
# Generate the embeddings for each chunks
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"
)
vectorstore = FAISS.from_documents(
    chunks , embeddings
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# define the retriever
retriever = vectorstore.as_retriever(
    search_kwargs = {'k' : 10} # get 10 most relavant chunks
)

In [ ]:
query = "Where is the A100 GPU used?"
docs = retriever.invoke(query)


for i in range(len(docs)):
    print(f"Chunk-{i}: \n{docs[i].page_content}\n")

Chunk-0: 
The company is famous for its 
graphics processing units.
pandas: 3 dogs: 2 giraffes: 4
Figure 13. In-context learning samples on company knowledge, object counting, and French poem. The predictions are from VILA-13B.
Q: Can you explain the meme? 
A: The meme depicts a man's reaction to the price of a computer 
graphics card. In the first image, the man is smiling and appears excited 
about the product. In the second image, he is shocked and disappointed

Chunk-1: 
B. Training Cost
We perform training on 16 A100 GPU nodes, each node
has 8 GPUs. The training hours for each stage of the 7B
model are: projector initialization: 4 hours; visual language
pre-training: 30 hours; visual instruction-tuning: 6 hours.
The training corresponds to a total of 5.1k GPU hours. Most
of the computation is spent on the pre-training stage.
We have not performed training throughput optimizations
like sample packing [32] or sample length clustering. We

Chunk-2: 
about the product. In the second i

In [9]:
# adding re-rank with cohere api
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
import os 
from dotenv import load_dotenv
load_dotenv()
cohere_api_key = os.getenv("COHERE_API_KEY")

# re-ranker retriever
compressor = CohereRerank(
    cohere_api_key = cohere_api_key,
    model = "rerank-english-v3.0"
)

In [10]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever = retriever,
    base_compressor = compressor
)

In [11]:
compressed_docs = compression_retriever.invoke(query)

In [12]:
len(compressed_docs)

3

In [13]:
for i , doc in enumerate(compressed_docs):
    print(f"Chunk-{i}")
    print(doc.page_content)

Chunk-0
B. Training Cost
We perform training on 16 A100 GPU nodes, each node
has 8 GPUs. The training hours for each stage of the 7B
model are: projector initialization: 4 hours; visual language
pre-training: 30 hours; visual instruction-tuning: 6 hours.
The training corresponds to a total of 5.1k GPU hours. Most
of the computation is spent on the pre-training stage.
We have not performed training throughput optimizations
like sample packing [32] or sample length clustering. We
Chunk-1
The company is famous for its 
graphics processing units.
pandas: 3 dogs: 2 giraffes: 4
Figure 13. In-context learning samples on company knowledge, object counting, and French poem. The predictions are from VILA-13B.
Q: Can you explain the meme? 
A: The meme depicts a man's reaction to the price of a computer 
graphics card. In the first image, the man is smiling and appears excited 
about the product. In the second image, he is shocked and disappointed
Chunk-2
The central challenge of unifying vision a

In [14]:
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    task = "text-generation",
    temperature = 0.8,
    max_new_tokens = 2000
)
model = ChatHuggingFace(llm = llm)

In [15]:
system_prompt = (
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise.\n\n"
    "{context}"
)

In [16]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
system_prompt = (
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [17]:
question_answer_chain = create_stuff_documents_chain(
    llm = model, prompt = prompt
)

In [18]:
# combine retriever with QA chain
from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever , question_answer_chain)

In [19]:
response = rag_chain.invoke({
    "input": query
})

In [20]:
response['answer']

'The A100 GPU is used in the training process, specifically on 16 nodes, each node having 8 GPUs, for a total of 128 GPUs.'

In [21]:
len(response['context'])

10